# Eventa: fine-tune Qwen2.5-Coder-1.5B for Node.js

Runs on a **free T4 GPU** (Kaggle: *Settings → Accelerator → GPU T4*, Colab: *Runtime → Change runtime type → T4*). Takes about 15–25 minutes.

1. Install Unsloth
2. Load the base model in 4-bit with LoRA adapters
3. Load `training/data/train.jsonl` from GitHub and render it with the Qwen chat template (the same system + user message the CLI sends)
4. Train on the assistant responses only
5. Export a Q4_K_M GGUF named `eventa-1.5b-q4_k_m.gguf`

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
BASE_MODEL = "unsloth/Qwen2.5-Coder-1.5B-Instruct"
REPO_RAW = "https://raw.githubusercontent.com/Jeeva1398/eventa/main/training"
MAX_SEQ_LEN = 4096
EPOCHS = 3
GGUF_NAME = "eventa-1.5b-q4_k_m.gguf"

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=1398,
)

In [ ]:
import json, urllib.request
from datasets import Dataset

def load_jsonl(url):
    text = urllib.request.urlopen(url).read().decode("utf-8")
    return [json.loads(line) for line in text.splitlines() if line.strip()]

def messages(row, with_answer=True):
    user = row["instruction"] + ("\n\n" + row["input"] if row["input"] else "")
    msgs = [{"role": "system", "content": row["system"]}, {"role": "user", "content": user}]
    if with_answer:
        msgs.append({"role": "assistant", "content": row["output"]})
    return msgs

def to_text(row):
    return {"text": tokenizer.apply_chat_template(messages(row), tokenize=False)}

train_rows = load_jsonl(f"{REPO_RAW}/data/train.jsonl")
eval_rows = load_jsonl(f"{REPO_RAW}/data/eval.jsonl")
train_ds = Dataset.from_list([to_text(r) for r in train_rows])
eval_ds = Dataset.from_list([to_text(r) for r in eval_rows])

lengths = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"]]
print(f"train={len(train_ds)} eval={len(eval_ds)} max_tokens={max(lengths)} mean_tokens={sum(lengths) // len(lengths)}")
train_ds = train_ds.filter(lambda r: len(tokenizer(r["text"])["input_ids"]) <= MAX_SEQ_LEN)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        eval_strategy="epoch",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=1398,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)

## Sanity check on held-out examples
These scenarios were never seen during training.

In [ ]:
FastLanguageModel.for_inference(model)

for row in eval_rows[:3]:
    prompt = tokenizer.apply_chat_template(messages(row, with_answer=False), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=400, temperature=0.2, do_sample=True)
    print("=" * 80, row["id"])
    print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Export GGUF (Q4_K_M)

In [ ]:
import glob, hashlib, os, shutil

model.save_pretrained_gguf("eventa-gguf", tokenizer, quantization_method="q4_k_m")
candidates = [p for p in glob.glob("**/*.gguf", recursive=True) if "q4_k_m" in p.lower() and not p.endswith(GGUF_NAME)]
shutil.move(candidates[0], GGUF_NAME)

sha = hashlib.sha256()
with open(GGUF_NAME, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        sha.update(chunk)
print(GGUF_NAME, os.path.getsize(GGUF_NAME), "bytes")
print("sha256", sha.hexdigest())

Download `eventa-1.5b-q4_k_m.gguf` (Kaggle: *Output* panel, Colab: the cell below) and put it in `training/`. Then evaluate it locally against the base model:

```bash
ollama create eventa -f training/Modelfile
npm run eval -w eventa-training -- --models qwen2.5-coder:1.5b,eventa
```

In [ ]:
try:
    from google.colab import files
    files.download(GGUF_NAME)
except ImportError:
    print(f"Kaggle: find {GGUF_NAME} under /kaggle/working in the Output panel")

## Publish to Hugging Face

This needs a Hugging Face **write** token (huggingface.co → Settings → Access Tokens). On Kaggle, add it under *Add-ons → Secrets* as `HF_TOKEN`; on Colab, under the key icon. The cell uploads the GGUF, the model card and the Ollama `Modelfile` to `jeeva1398/eventa-1.5b-gguf`. After that, `npx eventa model use eventa-1.5b` and `ollama run hf.co/jeeva1398/eventa-1.5b-gguf` work for everyone.

In [ ]:
import os, urllib.request
from getpass import getpass
from huggingface_hub import HfApi

HF_REPO = "jeeva1398/eventa-1.5b-gguf"

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name) or getpass(f"{name}: ")

api = HfApi(token=secret("HF_TOKEN"))
api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
for remote, local in [("README.md", "model-card.md"), ("Modelfile", "Modelfile")]:
    urllib.request.urlretrieve(f"{REPO_RAW}/{local}", local)
    api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=HF_REPO)
system_prompt = train_rows[0]["system"]
params = {"temperature": 0.2, "num_ctx": 8192, "stop": ["<|im_end|>", "<|endoftext|>"]}
api.upload_file(path_or_fileobj=system_prompt.encode(), path_in_repo="system", repo_id=HF_REPO)
api.upload_file(path_or_fileobj=json.dumps(params).encode(), path_in_repo="params", repo_id=HF_REPO)
api.upload_file(path_or_fileobj=GGUF_NAME, path_in_repo=GGUF_NAME, repo_id=HF_REPO)
print(f"https://huggingface.co/{HF_REPO}")